# Implementing ver 1 logic in python

I implemented the logic of the current AI player into python functions.  
I represented the board and moves using pytorch tensors. This wasn't necessary and was a learning curve but introduced some really cool optimisations in the calculations - as you can see many of the functions are just a few lines long.  
You can play vs the bot using the bottom 3 cells. (you will need to install pytorch)  
I now just need to write a minimax solver and search all game trees to ensure this bot always is optimal.  
I'm working on this testing rig because I had an idea of how to make a bot that works on arbitrary game boards using a couple heuristics like taking forks and preventing forks - but there are too many cases to work it out by hand so I needed an auto-tester.

In [3]:
import torch
import numpy as np

In [293]:
# Matrices representing the 8 lines on the board
# pretty proud of the compactness
each = torch.ones(3)
LD = torch.diag(each)
lines = [LD, LD.flip(0)] + [torch.outer(row, each) for row in LD] + [torch.outer(each, row) for row in LD]
lines


[tensor([[1., 0., 0.],
         [0., 1., 0.],
         [0., 0., 1.]]),
 tensor([[0., 0., 1.],
         [0., 1., 0.],
         [1., 0., 0.]]),
 tensor([[1., 1., 1.],
         [0., 0., 0.],
         [0., 0., 0.]]),
 tensor([[0., 0., 0.],
         [1., 1., 1.],
         [0., 0., 0.]]),
 tensor([[0., 0., 0.],
         [0., 0., 0.],
         [1., 1., 1.]]),
 tensor([[1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.]]),
 tensor([[0., 1., 0.],
         [0., 1., 0.],
         [0., 1., 0.]]),
 tensor([[0., 0., 1.],
         [0., 0., 1.],
         [0., 0., 1.]])]

In [ ]:
# checks if the positive player is within the specified moves of winning on the given line
def can_get_line_in(board, line, moves):
    pos_count = torch.sum(line*(board == 1).float())
    neg_count = torch.sum(line*(board == -1).float())
    return pos_count + moves >= 3 and neg_count == 0

In [ ]:
# checks if the positive player has won on any of the lines
def has_won(board):
    return any([can_get_line_in(board,line,0) for line in lines])

In [ ]:
# finds the available squares on lines which can be won in a single move
def find_winning_squares(board):
    squares = torch.zeros([3,3])
    for line in lines:
        squares += (board == 0).float() * line * can_get_line_in(board, line, 1).float()
        # print((board == 0).float() * line * can_get_line_in(board, line, 1).float())
    return squares

In [ ]:
# finds the highest priority valid submitted square - or the highest priority valid square if no valid squares were submitted

def validate(board, squares):
    # normalise signals
    squares = (squares != 0).float()

    # ensure square is free
    squares = squares * (board == 0).float()

    # if no valid moves select all moves
    if not squares.any():
        squares = torch.ones([3,3])

    # reinsure square is free
    squares = squares * (board == 0).float()

    # here is the square priority
    priority = torch.tensor([[9,5,7],
                             [4,3,2],
                             [8,1,6]]).float()

    # find the position of the highest priority square
    most_prioritised = torch.argmax((priority*squares).flatten())\
    
    # return the move in board form
    chosen_move = torch.zeros(9)
    chosen_move[most_prioritised] = 1.0
    return chosen_move.reshape([3,3])



    


In [ ]:
# I'm just following the boolean logic here basically exactly so as not to miss any subtle cases - that's why its so long
# makes the necessary strategic decisions to always win when possible and draw if not possible - given we have already checked for immediate wins and blocks
def long_term_strategy(board):
    squares = torch.zeros(3,3)
    if board[0][0] == 1:
        squares[2][2] = 1.
        if board[1][1] != 1:
            if not board[1][0]:
                squares[2][0] = 1
            if not board[0][1]:
                squares[0][2] = 1
    elif board[1][1] == 1:
        if board[0][0] == -1:
            squares[1][2] = 1.
            squares[2][1] = 1.
        
        if board[2][0] == -1:
            squares[1][2] = 1.
            squares[0][1] = 1.

        if board[0][2] == -1:
            squares[1][0] = 1.
            squares[2][1] = 1.

        if board[2][2] == -1:
            squares[1][0] = 1.
            squares[0][1] = 1.

        if not squares[1][0] and not squares[1][2]:
            
            if board[0][1] == -1:
                squares[0][0] = 1.
                squares[0][2] = 1.
            
            if board[1][0] == -1:
                squares[0][0] = 1.
                squares[2][0] = 1.
                
            if board[2][1] == -1:
                squares[2][0] = 1.
                squares[2][2] = 1.
            
            if board[1][2] == -1:
                squares[0][2] = 1.
                squares[2][2] = 1.


    else:
        squares[1][1] = 1
        if not (squares * torch.tensor([[1,1,1],[1,0,1],[1,1,1]]).float()).any():
            squares[0][0] = 1

    return squares

In [ ]:
# returns the best move by checking for wins, then blocks, then long term strategy
def find_best(board, turn):
    board = board * turn
    winners = find_winning_squares(board)
    blockers = find_winning_squares(-board)
    if winners.any():
        return validate(board, winners)
    if blockers.any():
        return validate(board, blockers)
    return validate(board, long_term_strategy(board))

In [ ]:
# plays the provided move on the board and inverts the turn counter - used for testing
def play_move(board, move, turn):
    if (board * move).any():
        print("invalid move")
        return
    board += move*turn
    turn *= -1
    return board

In [274]:
board = torch.tensor([[1,0,0],[-1,0,0],[0,0,0]]).float()
board

tensor([[ 1.,  0.,  0.],
        [-1.,  0.,  0.],
        [ 0.,  0.,  0.]])

In [275]:
find_best(board,1)

tensor([[0., 0., 1.],
        [0., 0., 0.],
        [0., 0., 0.]])

In [276]:
print(long_term_strategy(board))

tensor([[0., 0., 1.],
        [0., 0., 0.],
        [0., 0., 1.]])


In [287]:
board = torch.zeros([3,3])
turn = torch.tensor(1)

In [292]:
play_move(board, find_best(board, turn), turn)

tensor([[ 1.,  0.,  1.],
        [-1.,  0.,  0.],
        [ 1., -1.,  0.]])

In [291]:
player_move = torch.tensor([[0,0,0],
                            [1,0,0],
                            [0,0,0]]).float()
play_move(board, player_move, turn)

tensor([[ 1.,  0.,  0.],
        [-1.,  0.,  0.],
        [ 1., -1.,  0.]])